# Episode 23: Muon vs AdamW, Lion and Sophia: a fair race

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep023-optimizers.ipynb)

We write AdamW from scratch, then race it against Lion, Sophia and Muon on our stories model, each with its own sweep of the learning rate, and measure the steps, the time and the memory each needs.

[The video](https://youtu.be/0ysdFCrmdlU) · [The episode's page](https://mvgpt.nenuadrian.com/ep023-optimizers/) · [The docs](https://nenuadrian.github.io/mvGPT/optimizers/) · [The code at `ep023/end`](https://github.com/nenuadrian/mvGPT/tree/ep023/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep023/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

## The tests
Our AdamW and Muon against PyTorch's, and what each update does.

In [ ]:
!python -m pytest tests/test_optimizers.py -v

## A run of the race
The stories model, 1,000 steps with AdamW at a peak learning rate of 2.4e-3, on the BPE
stories.

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids
!python bakeoff.py adamw 2.4e-3

## The whole race
The episode ran 35 of these: each optimizer at several learning rates, the best with three
seeds, and AdamW at other lengths. Here are its runs from the Hub, in `results/bakeoff`,
next to yours.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("nenuadrian/mvGPT-checkpoints", allow_patterns="results/bakeoff/*", local_dir=".")

In [ ]:
!python bakeoff_report.py adamw
!python bakeoff_report.py
!python bakeoff_report.py --best

Next: [episode 24, Text to Cypher: our GPT asks a graph database](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep024-knowledge-graph.ipynb)